In [14]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import CreateTable, CreateTableColumn, CreateTableConstraints, CreateForeignKey
import pandas as pd
from pandas.core.interchange.dataframe_protocol import DataFrame
from dotenv import load_dotenv
import os 
from dbrepo.api.dto import CreateView
from dbrepo.api.dto import CreateView, Subset, SubsetColumn, Join
from dbrepo.api.dto import JoinType

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [15]:
container_id = '6cfb3b8e-1792-4e46-871a-f3d103527203'
DB_ID = os.getenv("DB_ID")

In [ ]:
from dbrepo.api.dto import (
    CreateIdentifier,
    CreateIdentifierTitle,
    CreateIdentifierDescription,
    RelatedIdentifierType,
    RelatedIdentifierRelation,
    CreateIdentifierCreator,
    CreateRelatedIdentifier,
    DescriptionType,
    IdentifierType,
    License,
    Language
)

def build_consolidated_use_case_pid(database_id) -> CreateIdentifier:
    """
    Generates DBRepo metadata
    """
    
    titles = [
        CreateIdentifierTitle(
            title="Predictive Modeling of Regional GDP based on Wastewater-Based Epidemiology",
            language=Language.EN
        )
    ]
    
    # 2. Comprehensive Metadata Descriptions (Abstract, Methods, Scope, Units)
    descriptions = [
        CreateIdentifierDescription(
            description=(
                """Abstract: This use case explores the correlation relationship between 
                illicit drug use in major European cities and their regional economic productivity (GDP). 
                Original Publishers: EUDA & SCORE, EUROSTAT.
                EUDA & SCORE: URI: https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en
                EUROSTAT: DOI: https://doi.org/10.2908/NAMA_10R_3GDP, URI: https://ec.europa.eu/eurostat/databrowser/view/nama_10r_3gdp__custom_20659344/default/table"""
            ),
            type=DescriptionType("Abstract"),
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Data Stewardship and Preprocessing Challenge: While the drug dataset identifies locations 
                by specific city strings (e.g., 'Graz', 'Steyr'), Eurostat utilizes standardized NUTS-3 administrative codes 
                (e.g., 'DE212'). This is resolved via a custom mapping schema table ('city_map').
                Only the active filtered subset utilized in this longitudinal frame is republished here."""
            ),
            type="Methods",
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Temporal & Spatial Coverage: 
                Annual wastewater tracking spans years 2011 to 2025 across 115 cities and 25 countries in the European Union, 
                Norway, and Türkiye. GDP tracking spans annually from 2000 to 2024 across EU Member States, Candidate and 
                potential Candidate Countries, Norway, and Switzerland. Our subset only contains common European countries 
                for the years 2011-2024. Data availability varies across years and regions."""
            ),
            type=DescriptionType("TechnicalInfo"),
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Units of Measure: "
                Wastewater metrics indicate concentrations (mg/1000p/day) of illicit drug loads (Cocaine, Methamphetamine, MDMA) 
                measured from 24-hour composite samples collected over a single week between March and May. 
                GDP values indicate economic output expressed in Euros at current market prices by NUTS 3 region."""
            ),
            type=DescriptionType("Other"),
            language=Language.EN
        )
    ]
    
    # 3. Explicit Provenance & Lineage Relationships
    related_identifiers = [
        # Upstream Eurostat Source Dataset
        CreateRelatedIdentifier(
            id="10.2908/NAMA_10R_3GDP",
            value="10.2908/NAMA_10R_3GDP",
            type=RelatedIdentifierType.DOI,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        ),
        # Upstream EUDA Open Repository Source
        CreateRelatedIdentifier(
            id="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            value="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            type=RelatedIdentifierType.URL,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        )
    ]

    cc_by_4_0 = License(
        identifier="CC-BY-4.0",
        uri="https://creativecommons.org/licenses/by/4.0/",
        description=(
            "Creative Commons Attribution 4.0 International: Allows users to copy, "
            "distribute, display, perform, and modify the work, even for commercial purposes, "
            "provided that they give appropriate credit to the original creator."
        )
    )
    
    # 4. Master DataCite Payload Assembly
    identifier_payload = CreateIdentifier(
        database_id=database_id,
        #table_id = 'dbac11e2-536e-49a1-8147-1add3b0bb10d',
        publication_year=2026,           # Project release date
        publisher="EUDA & SCORE, EUROSTAT",
        type = IdentifierType.DATABASE,
        language=Language.EN,
        licenses=[cc_by_4_0],          # Explicitly declared open license for both sources
        titles=titles,
        descriptions=descriptions,
        related_identifiers=related_identifiers,
        funders= [],
        
        # Comprehensive project curator/author roster mapped to standard DataCite creator formats
        creators=[
            CreateIdentifierCreator(creator_name="Helene Vaught", firstname= "Helene", lastname= "Vaught", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Vlada Hlushchenko", firstname= "Vlada", lastname= "Hlushchenko", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Barnabás Paksi", firstname= "Barnabás", lastname= "Paksi", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Amélie Assmayr", firstname= "Amélie", lastname= "Assmayr", affiliation= "TU Wien")
        ]
    )
    
    return identifier_payload

In [17]:
identifier_payload = build_consolidated_use_case_pid(database_id=DB_ID)

In [18]:
response = client._wrapper(
    method="post", 
    url=f'/api/v1/identifier', 
    payload=identifier_payload
)

response.raise_for_status()

In [19]:
tables = client.get_tables(DB_ID)

tab_ids = dict()
for t in tables:
    if t.name == "gdp_data":
        tab_ids["gdp_data"] = t.id 
    if t.name == "wastewater_data":
        tab_ids["wastewater_data"] = t.id 
    if t.name == "city_map":
        tab_ids["city_map"] = t.id 

In [20]:
def wastewater_metadata(DB_ID, tab_id):
    titles = [
            CreateIdentifierTitle(
                title="Wastewater Metabolite Concentrations",
                language=Language.EN
            )
        ]
        
    descriptions = [
        CreateIdentifierDescription(
            description="Estimated concentrations of metabolites in municipal wastewater for various cities over the period of 2011-2024. Subset from EUDA and SCORE",
            type=DescriptionType("Abstract"),
            language=Language.EN
        )
    ]

    related_identifiers = [
        CreateRelatedIdentifier(
                                value="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
                                type=RelatedIdentifierType.URL,
                                relation=RelatedIdentifierRelation.IS_DERIVED_FROM
                                )
    ]

    cc_by_4_0 = License(
        identifier="CC-BY-4.0",
        uri="https://creativecommons.org/licenses/by/4.0/",
        description=(
            "Creative Commons Attribution 4.0 International: Allows users to copy, "
            "distribute, display, perform, and modify the work, even for commercial purposes, "
            "provided that they give appropriate credit to the original creator."
        )
    )
        
        
    identifier_payload = CreateIdentifier(
        database_id=DB_ID,
        table_id=tab_id,
        publication_year=2026,           # Project release date
        publisher="EUDA, SCORE",
        type = IdentifierType.TABLE,
        language=Language.EN,
        licenses=[cc_by_4_0],          # Explicitly declared open license for all sources
        titles=titles,
        descriptions=descriptions,
        related_identifiers=related_identifiers,
        funders= [],
        
        # Comprehensive project curator/author roster mapped to standard DataCite creator formats
        creators=[
            CreateIdentifierCreator(creator_name="Helene Vaught", firstname= "Helene", lastname= "Vaught", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Vlada Hlushchenko", firstname= "Vlada", lastname= "Hlushchenko", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Barnabás Paksi", firstname= "Barnabás", lastname= "Paksi", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Amélie Assmayr", firstname= "Amélie", lastname= "Assmayr", affiliation= "TU Wien")
        ]
    )

    return identifier_payload

ww_payload = wastewater_metadata(DB_ID, tab_ids["wastewater_data"])

In [21]:
def gdp_metadata(DB_ID, tab_id):
    titles = [
            CreateIdentifierTitle(
                title="Regional GDP 2011-2024 at Present Values",
                language=Language.EN
            )
        ]
        
    descriptions = [
        CreateIdentifierDescription(
            description="This table stores the economic baseline for European regions (gross domestic product at current market prices by NUTS3 regions). Sourced from Eurostat.",
            type=DescriptionType("Abstract"),
            language=Language.EN
        )
    ]

    related_identifiers = [
        CreateRelatedIdentifier(
                                value="10.2908/NAMA_10R_3GDP",
                                type=RelatedIdentifierType.DOI,
                                relation=RelatedIdentifierRelation.IS_DERIVED_FROM
                                )
    ]

    cc_by_4_0 = License(
        identifier="CC-BY-4.0",
        uri="https://creativecommons.org/licenses/by/4.0/",
        description=(
            "Creative Commons Attribution 4.0 International: Allows users to copy, "
            "distribute, display, perform, and modify the work, even for commercial purposes, "
            "provided that they give appropriate credit to the original creator."
        )
    )
        
        
    identifier_payload = CreateIdentifier(
        database_id=DB_ID,
        table_id=tab_id,
        publication_year=2026,           # Project release date
        publisher="Eurostat",
        type = "table", # subset does not work
        language=Language.EN,
        licenses=[cc_by_4_0],          # Explicitly declared open license for all sources
        titles=titles,
        descriptions=descriptions,
        related_identifiers=related_identifiers,
        funders= [],
        
        # Comprehensive project curator/author roster mapped to standard DataCite creator formats
        creators=[
            CreateIdentifierCreator(creator_name="Helene Vaught", firstname= "Helene", lastname= "Vaught", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Vlada Hlushchenko", firstname= "Vlada", lastname= "Hlushchenko", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Barnabás Paksi", firstname= "Barnabás", lastname= "Paksi", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Amélie Assmayr", firstname= "Amélie", lastname= "Assmayr", affiliation= "TU Wien")
        ]
    )

    return identifier_payload

gdp_payload = gdp_metadata(DB_ID, tab_ids["gdp_data"])

In [22]:
def city_map_metadata(DB_ID, tab_id):
    titles = [
            CreateIdentifierTitle(
                title="City Name to NUTS-3 Code Mapping",
                language=Language.EN
            )
        ]
        
    descriptions = [
        CreateIdentifierDescription(
            description="This table serves as the bridge/mapping schema. It resolves the city names used by the EUDA to the NUTS-3 codes used by Eurostat.",
            type=DescriptionType("Abstract"),
            language=Language.EN
        )
    ]

    related_identifiers = []

    cc_by_4_0 = License(
        identifier="CC-BY-4.0",
        uri="https://creativecommons.org/licenses/by/4.0/",
        description=(
            "Creative Commons Attribution 4.0 International: Allows users to copy, "
            "distribute, display, perform, and modify the work, even for commercial purposes, "
            "provided that they give appropriate credit to the original creator."
        )
    )
        
        
    identifier_payload = CreateIdentifier(
        database_id=DB_ID,
        table_id=tab_id,
        publication_year=2026,           # Project release date
        publisher="TU Wien",
        type = "table", # subset does not work
        language=Language.EN,
        licenses=[cc_by_4_0],          # Explicitly declared open license for all sources
        titles=titles,
        descriptions=descriptions,
        related_identifiers=related_identifiers,
        funders= [],
        
        # Comprehensive project curator/author roster mapped to standard DataCite creator formats
        creators=[
            CreateIdentifierCreator(creator_name="Helene Vaught", firstname= "Helene", lastname= "Vaught", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Vlada Hlushchenko", firstname= "Vlada", lastname= "Hlushchenko", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Barnabás Paksi", firstname= "Barnabás", lastname= "Paksi", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Amélie Assmayr", firstname= "Amélie", lastname= "Assmayr", affiliation= "TU Wien")
        ]
    )

    return identifier_payload

city_payload = city_map_metadata(DB_ID, tab_ids["city_map"])

In [23]:
response = client._wrapper(
    method="post", 
    url=f'/api/v1/identifier', 
    payload=gdp_payload
)

response.raise_for_status() 

In [24]:
response = client._wrapper(
    method="post", 
    url=f'/api/v1/identifier', 
    payload=ww_payload
)

response.raise_for_status() 

In [25]:
response = client._wrapper(
    method="post", 
    url=f'/api/v1/identifier', 
    payload=city_payload
)

response.raise_for_status() 